In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow.keras.models import load_model


In [ ]:
ROOT_DIR = Path.cwd().parents[1]
sys.path.append(str(ROOT_DIR / "src" / "data_preprocessing"))
from normalize_fn import load

labels_file = r"C:\Users\mayak\DeepSyn\datasets\labels.csv"
model_file  = r"C:\Users\mayak\DeepSyn\src\model_training\checkpoints\final_model.h5"
output_file = "model_predictions.csv"

In [ ]:
model = load_model(model_file)
print("Model input shape:", model.input_shape)

In [ ]:
X_tr, X_val, X_train, X_test, y_tr, y_val, y_train, y_test = load(norm='norm')
print("X_tr shape:",    X_tr.shape)
print("X_train shape:", X_train.shape)
print("X_test shape:",  X_test.shape)
X_all = np.vstack([X_train, X_test])
print("X_all shape:", X_all.shape)

In [ ]:
expected_features = model.input_shape[1]
if X_all.shape[1] != expected_features:
    raise ValueError(
        f"Feature mismatch! X_all has {X_all.shape[1]} features, "
        f"model expects {expected_features}. "
        f"Make sure normalize_fn.py has been updated with the fix."
    )
print("Feature count matches model input.")

In [ ]:
batch_size = 1024
y_all_pred = model.predict(X_all, batch_size=batch_size).squeeze()
print("Predictions shape:", y_all_pred.shape)

In [ ]:
labels_df = pd.read_csv(labels_file)
labels_train = labels_df[labels_df['fold'] != 0]
labels_test  = labels_df[labels_df['fold'] == 0]
labels_df_ordered = pd.concat([labels_train, labels_test], axis=0).reset_index(drop=True)

In [ ]:
if len(labels_df_ordered) != len(y_all_pred):
    raise ValueError(
        f"Row mismatch! labels has {len(labels_df_ordered)} rows "
        f"but predictions has {len(y_all_pred)} values."
    )

In [ ]:
labels_df_ordered['ID'] = (
    labels_df_ordered['drug_a_name'] + "_" +
    labels_df_ordered['drug_b_name'] + "_" +
    labels_df_ordered['cell_line']
)
labels_df_ordered['predicted_synergy'] = y_all_pred.round(8)

pred_df = labels_df_ordered[['ID', 'drug_a_name', 'drug_b_name', 'cell_line', 'synergy', 'predicted_synergy']]

In [ ]:
pred_df.to_csv(output_file, index=False)
print(f"Predictions saved to: {output_file}")
print(pred_df.head())